In [1]:
import logging
from datetime import datetime

import pytest
from fastapi import FastAPI, HTTPException
from fastapi.exceptions import RequestValidationError
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from sqlalchemy.exc import SQLAlchemyError
from starlette.exceptions import HTTPException as StarletteHTTPException

from llm_app.core.exceptions import AppError, NotFoundError
from llm_app.server.exception_handlers import (
    app_error_handler,
    http_exception_handler,
    sqlalchemy_exception_handler,
    unhandled_exception_handler,
    validation_exception_handler,
)
from llm_app.server.middleware import RequestContextMiddleware


In [2]:
class Payload(BaseModel):
    username: str = Field(min_length=1)
    password: str = Field(min_length=8)


@pytest.fixture
def app() -> FastAPI:
    app = FastAPI()
    app.add_middleware(RequestContextMiddleware)

    app.add_exception_handler(AppError, app_error_handler)
    app.add_exception_handler(RequestValidationError, validation_exception_handler)
    app.add_exception_handler(StarletteHTTPException, http_exception_handler)
    app.add_exception_handler(SQLAlchemyError, sqlalchemy_exception_handler)
    app.add_exception_handler(Exception, unhandled_exception_handler)

    # 下面是测试专用的"故障路由"
    @app.get('/app-error')
    def app_error() -> None:
        raise NotFoundError(
            '会话不存在',
            details={'session_id': 'abc'},
            headers={'X-Test': '1'},
        )

    @app.post('/validation')
    def validation(payload: Payload) -> dict[str, bool]:
        return {'ok': True}

    @app.get('/http-401')
    def http_401() -> None:
        raise HTTPException(401, detail='未认证', headers={'WWW-Authenticate': 'Bearer'})

    @app.get('/http-500')
    def http_500() -> None:
        raise HTTPException(500, detail='内部秘密，不能给客户端看')

    @app.get('/db-error')
    def db_error() -> None:
        raise SQLAlchemyError('password=supersecret')

    @app.get('/boom')
    def boom() -> None:
        raise RuntimeError('password=supersecret')

    return app


@pytest.fixture
def client(app: FastAPI) -> TestClient:
    # 关键：不让 TestClient 把服务端异常直接抛出来，我们要看 500 响应体
    return TestClient(app, raise_server_exceptions=False)


In [4]:
# notebook 中没有 pytest 的 fixture 注入，需要手动解包并执行 fixture 原函数
test_app = app.__wrapped__()
client = client.__wrapped__(test_app)

resp = client.get('/app-error', headers={'X-Request-ID': 'rid-test-110'})
body = resp.json()

resp.status_code, dict(resp.headers), body

会话不存在


(404,
 {'x-test': '1',
  'x-request-id': 'rid-test-110',
  'content-length': '154',
  'content-type': 'application/json'},
 {'code': 'NOT_FOUND',
  'message': '会话不存在',
  'request_id': 'rid-test-110',
  'timestamp': '2026-11-24T03:35:51.302451+00:00',
  'details': {'session_id': 'abc'}})

In [ ]:
caplog = pytest.LogCaptureFixture(client)
with caplog.at_level(logging.WARNING, logger='llm_app.server.exception_handlers'):
    client.get('/app-error')